# 📸🎬 寫實人像生圖 & 照片轉動態影片工作站 (L4 / A100 專用)

這份專屬筆記本整合了你上傳的兩張參考圖（`ref_1.jpg`、`ref_2.jpg`），並提供兩大核心功能：
1. **同風格寫實人像生圖（Text-to-Image & Image-to-Image）**：使用頂級寫實模型 `RealVisXL V5.0`，提供「**對齊附件白床晨光風格**」與「**原始和室榻榻米全身鏡風格**」雙版本優化 Prompt，並支援以你的參考圖為基底延伸新動作與姿勢。
2. **讓照片真正動起來（Wan 2.1 14B Image-to-Video 圖生影片）**：直接將你的 `ref_1.jpg`、`ref_2.jpg`（或新生成的照片）送入 **Wan 2.1 I2V 14B** 旗艦圖生影模型，生成 5 秒自然呼吸、眨眼、微轉頭互動與柔和運鏡的真實動態影片！

> 💡 **硬體建議**：因為 Step 4 使用 **Wan 2.1 I2V 14B** 旗艦圖生影片模型，強烈建議在 Colab 選擇 **`A100 GPU`** 並勾選 **「大量 RAM (High-RAM)」** 以獲得最佳畫質與最快速度。

## Step 1：自動從 GitHub 載入你的參考圖並檢測 A100 / L4 硬體

In [ ]:
import os
import torch
import psutil
from PIL import Image
from IPython.display import display

# 自動同步 GitHub 儲存庫內的參考圖片 (免手動上傳)
if not os.path.exists("colab-ai-media-lab"):
    !git clone -q https://github.com/Chenponpon/colab-ai-media-lab.git
else:
    !cd colab-ai-media-lab && git pull -q

REF_IMG_1 = "colab-ai-media-lab/assets/ref_1.jpg"
REF_IMG_2 = "colab-ai-media-lab/assets/ref_2.jpg"

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f"✅ GPU: {device_name} | VRAM: {vram_gb:.2f} GB | System RAM: {ram_gb:.2f} GB")
else:
    print("❌ 未偵測到 GPU！請點選上方『執行階段 -> 變更執行階段類型』切換為 A100 GPU 或 L4 GPU。")

print("\n📸 已成功載入你的兩張參考照片 (ref_1.jpg & ref_2.jpg)：")
img1 = Image.open(REF_IMG_1)
img2 = Image.open(REF_IMG_2)
preview = Image.new('RGB', (img1.width + img2.width + 20, max(img1.height, img2.height)), (255, 255, 255))
preview.paste(img1, (0, 0))
preview.paste(img2, (img1.width + 20, 0))
preview.thumbnail((800, 600))
display(preview)

## Step 2：安裝最新版 Diffusers 與影片編碼套件

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf imageio imageio-ffmpeg ftfy

## Step 3：產出相同風格的寫實照片 (支援「純文字生圖」與「參考圖延伸生圖」)

這裡我幫你整理並優化了兩組 Prompt：
- **`STYLE_A_BEDROOM`（精準還原你上傳的兩張照片風格）**：三位混血女孩（長直黑髮、深棕短鮑伯頭、長捲褐髮），在純白床單與極簡灰白牆前、柔和自然窗光下的底片感寫真。
- **`STYLE_B_WASHITSU`（優化後的原始日式和室榻榻米 + 背後全身鏡 Prompt）**：強化了攝影鏡頭參數與自然膚質細節描述。

In [ ]:
import gc
import time
import torch
from PIL import Image
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image, DPMSolverMultistepScheduler
from IPython.display import display

# ==================== 參數調整區 ====================
# 是否開啟「保留核心人設 + 多樣化隨機抽樣」(True = 每張圖自動隨機變換臉型、髮色、髮型、妝容、姿勢、服裝款式/顏色與背景)
RANDOMIZE_EVERY_IMAGE = True

# 若關閉隨機抽樣 (RANDOMIZE_EVERY_IMAGE = False)，則可選擇固定風格:
PROMPT_STYLE = "STYLE_A_BEDROOM"  # "STYLE_A_BEDROOM" 或 "STYLE_B_WASHITSU"

# 是否以附件照片作為構圖基底 (若開啟隨機姿勢與背景，建議設為 False 讓畫面自由發揮！)
USE_REF_IMAGE_AS_BASE = False
BASE_IMAGE_PATH = REF_IMG_1
DENOISING_STRENGTH = 0.65
NUM_IMAGES = 4               # 一次生成 4 張不同隨機組合的照片
# ====================================================

import sys
sys.path.insert(0, "colab-ai-media-lab/scripts")
from prompt_randomizer import generate_diversified_prompt

PROMPTS = {
    "STYLE_A_BEDROOM": (
        "RAW candid documentary-style medium-full portrait photograph on 35mm Kodak Portra 400 film, "
        "three real 20-year-old French-Vietnamese Eurasian women with distinctly different faces sitting closely together on crisp white linen bedsheets against a minimalist warm off-white wall. "
        "Left woman has long straight jet-black hair; center woman has a cute chin-length dark brown bob haircut with subtle freckles and rosy cheeks; "
        "right woman has long wavy chestnut brown hair. "
        "Extremely fair translucent skin with natural rosy flush, visible skin pores, soft realistic skin texture. "
        "Slim hourglass proportions, natural soft full breasts with realistic gravity and shape, slim waists. "
        "Wearing delicate sheer floral lace and mesh bodysuits and lingerie in muted pastel pink, nude taupe, and black. "
        "Soft diffused morning window light from the side, gentle shadows, unedited snapshot aesthetic, 85mm f/2.0 lens, ultra-detailed."
    ),
    "STYLE_B_WASHITSU": (
        "RAW documentary-style full-body photograph on 35mm film of three real 20-year-old French-Vietnamese mixed-race women "
        "with distinctly different facial features standing close together on Japanese tatami mats in a traditional washitsu room. "
        "Left woman with long straight black hair, center woman with short dark brown bob, right woman with long wavy dark brown hair. "
        "Extremely fair translucent skin, natural rosy undertone, subtle realistic skin pores and texture. "
        "Slim hourglass bodies, 165cm 50kg proportions, natural soft full F-cup breasts with realistic shape, slim waists. "
        "Interacting naturally, light touches on shoulders, shared soft smiles, full body visible head to toe. "
        "Large clear mirror behind them reflecting their backs. Wearing different sheer translucent tulle mesh and delicate lace lingerie in white, black, and soft pink. "
        "Soft natural daylight through shoji screen, photorealistic, unedited snapshot feel."
    )
}

NEGATIVE_PROMPT = (
    "cgi, 3d render, plastic skin, airbrushed, doll, mannequin, anime, cartoon, "
    "deformed hands, extra fingers, mutated limbs, bad anatomy, fused bodies, identical faces, "
    "overexposed, harsh flash, watermark, text, blurry, low quality"
)

gc.collect()
torch.cuda.empty_cache()

model_id = "SG161222/RealVisXL_V5.0"
print(f"⏳ 正在載入寫實人像模型 {model_id}...")

if USE_REF_IMAGE_AS_BASE:
    pipe = AutoPipelineForImage2Image.from_pretrained(
        model_id, torch_dtype=torch.float16, variant="fp16"
    ).to("cuda")
else:
    pipe = AutoPipelineForText2Image.from_pretrained(
        model_id, torch_dtype=torch.float16, variant="fp16"
    ).to("cuda")

pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++"
)

for i in range(NUM_IMAGES):
    start_t = time.time()
    if RANDOMIZE_EVERY_IMAGE:
        active_prompt, recipe = generate_diversified_prompt()
        print(f"\n🎲 [第 {i+1} 張隨機組合] 姿勢: {recipe['pose'][:45]}... | 背景: {recipe['background'][:45]}...")
    else:
        active_prompt = PROMPTS[PROMPT_STYLE]

    if USE_REF_IMAGE_AS_BASE:
        init_img = Image.open(BASE_IMAGE_PATH).convert("RGB").resize((832, 1216), Image.LANCZOS)
        img = pipe(
            prompt=active_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            image=init_img,
            strength=DENOISING_STRENGTH,
            num_inference_steps=30,
            guidance_scale=5.5
        ).images[0]
    else:
        img = pipe(
            prompt=active_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            width=832,
            height=1216,
            num_inference_steps=30,
            guidance_scale=5.5
        ).images[0]
    
    out_name = f"generated_portrait_{i+1}.jpg"
    img.save(out_name, quality=95)
    generated_files.append(out_name)
    print(f"✅ 第 {i+1} 張生圖完成 ({out_name}) | 耗時: {time.time() - start_t:.1f} 秒")
    display(img)

# 釋放生圖模型顯存，讓接下來的 14B 圖生影片模型全力運行
del pipe
gc.collect()
torch.cuda.empty_cache()

## Step 4：讓照片動起來！Wan 2.1 (14B) Image-to-Video 照片轉動態影片

這一步會直接把你上傳的 `ref_1.jpg` 或 `ref_2.jpg`（也可以改成上面 Step 3 剛算好的新照片）轉成一段 **5 秒（81 影格）的自然動態影片**！
- 模型會保留照片中三位女孩的臉孔、髮型與服裝細節，並加入自然的呼吸起伏、眼神眨動、微笑與微轉頭互動。

In [ ]:
import gc
import time
import torch
import numpy as np
from PIL import Image
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video
from transformers import CLIPVisionModel
from IPython.display import HTML, display
from base64 import b64encode

# ==================== 影片生成設定區 ====================
# 要讓哪幾張照片動起來？預設同時把你的兩張附件照片 (REF_IMG_1, REF_IMG_2) 轉成影片！
# 若想把 Step 3 新生成的照片也轉成影片，可加上 "generated_portrait_1.jpg"
IMAGES_TO_ANIMATE = [REF_IMG_1, REF_IMG_2]

# 動作描述 Prompt (指導 AI 如何讓畫面中的人物自然動起來)
MOTION_PROMPT = (
    "Natural candid video of three young women sitting closely together on a soft white bed. "
    "Subtle realistic breathing movements, gentle natural eye blinks, warm soft smiles, "
    "leaning slightly toward one another and interacting softly, hair gently moving, "
    "soft morning window light, smooth subtle handheld camera sway, photorealistic, high skin detail."
)

NEGATIVE_VIDEO_PROMPT = (
    "bright tones, overexposed, static, blurred details, subtitles, watermark, "
    "distorted face, morphing features, extra limbs, deformed hands, sudden jerky movement, worst quality"
)

NUM_FRAMES = 81   # 81 影格 @ 16fps ≈ 5 秒完整動態影片 (若想先快速測試可改為 49)
STEPS = 30        # 推論步數 (25~30 畫質最佳)
# ========================================================

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_id = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
print(f"⏳ 正在載入旗艦級圖生影片模型 {model_id} (首次執行需下載權重，請稍候)...")

image_encoder = CLIPVisionModel.from_pretrained(model_id, subfolder="image_encoder", torch_dtype=torch.float32)
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(
    model_id, vae=vae, image_encoder=image_encoder, torch_dtype=torch.bfloat16
)

# 啟用智慧顯存管理與 VAE Tiling，確保在 A100 40GB / L4 上穩定輸出長影格不爆顯存
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

# 直式人像最佳解析度 (必須為 16 的倍數：544 x 832 完美匹配你的 2:3 照片比例)
TARGET_W, TARGET_H = 544, 832

for idx, img_path in enumerate(IMAGES_TO_ANIMATE):
    print(f"\n🎬 [{idx+1}/{len(IMAGES_TO_ANIMATE)}] 正在將 {img_path} 轉換為動態影片 ({TARGET_W}x{TARGET_H}, {NUM_FRAMES} 影格)...")
    input_image = Image.open(img_path).convert("RGB").resize((TARGET_W, TARGET_H), Image.LANCZOS)
    
    start_t = time.time()
    output_frames = pipe(
        image=input_image,
        prompt=MOTION_PROMPT,
        negative_prompt=NEGATIVE_VIDEO_PROMPT,
        height=TARGET_H,
        width=TARGET_W,
        num_frames=NUM_FRAMES,
        guidance_scale=5.0,
        num_inference_steps=STEPS
    ).frames[0]
    
    elapsed = time.time() - start_t
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    out_video = f"animated_output_{idx+1}.mp4"
    export_to_video(output_frames, out_video, fps=16)
    
    print(f"✅ 影片生成完成：{out_video} | 耗時: {elapsed:.1f} 秒 | 峰值 VRAM: {peak_vram:.2f} GB")
    mp4_bytes = open(out_video, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
    display(HTML(f'<video width=420 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))

## Step 5：一鍵打包下載所有生成的圖片與 MP4 影片到電腦

In [ ]:
import glob
import zipfile
from google.colab import files

zip_filename = "ai_portraits_and_videos.zip"
with zipfile.ZipFile(zip_filename, 'w') as zf:
    for f in glob.glob("generated_portrait_*.jpg") + glob.glob("animated_output_*.mp4"):
        zf.write(f)
        print(f"📦 已加入壓縮檔: {f}")

files.download(zip_filename)